In [1]:
"""
Preprocess and scale CICIDS2017 (source domain). Save scaler for reuse on target
dataset, and calculate covariance statistics.
"""

### Imports ###
import json
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.covariance import LedoitWolf

# Load shared feature-space artifacts in a single, validated format.
def load_feature_order(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    if isinstance(payload, dict):
        if "features" not in payload:
            raise ValueError(
                f"Expected key 'features' in {path} when JSON object is provided."
            )
        feature_order = list(payload["features"])
    elif isinstance(payload, list):
        feature_order = list(payload)
    else:
        raise ValueError(
            f"Unsupported shared feature space format in {path}: "
            f"{type(payload).__name__}"
        )

    if not feature_order:
        raise ValueError(f"Shared feature space in {path} is empty")

    return feature_order

In [2]:
### Import and concatenate CSVs ###

# Creates a Path object pointing to the source-domain CSV directory.
data_dir = Path("data/raw/source")

# Read each CSV with encoding fallback for files that are not UTF-8.
def read_csv_with_fallback(file_path):
    for enc in ("utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(file_path, low_memory=False, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError("unknown", b"", 0, 1, f"Unable to decode {file_path}")

# Load all source CSV files into a list of DataFrames.
dfs = [read_csv_with_fallback(f) for f in data_dir.glob("*.csv")]

# Concatenate all DataFrames into one source-domain DataFrame.
df = pd.concat(dfs, ignore_index=True)

# Display a quick shape check and preview rows.
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (2830743, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,22,166,1,1,0,0,0,0,0.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
1,60148,83,1,2,0,0,0,0,0.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
2,123,99947,1,1,48,48,48,48,48.0,0.0,...,40,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
3,123,37017,1,1,48,48,48,48,48.0,0.0,...,32,0.000,0.000,0,0,0.0,0.000,0,0,BENIGN
4,0,111161336,147,0,0,0,0,0,0.0,0.0,...,0,1753752.625,2123197.578,4822992,95,9463032.7,2657727.996,13600000,5700287,BENIGN


In [3]:
### [Diagnostic] Extract features and labels ###

# Feature space: list every column present in the compiled source dataset.
feature_columns = list(df.columns)
print(f"Feature/column count: {len(feature_columns)}")
print("Feature space (all dataset columns):")
for column_name in feature_columns:
    print(f"- {column_name}")

# Label space: detect the raw label column robustly before sanitization trims names.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_feature = next((column_name for column_name in LABEL_CANDIDATES if column_name in df.columns), None)
if label_feature is None:
    available_label_like_columns = [
        column_name
        for column_name in df.columns
        if column_name.strip().lower() in {"label", "attack"}
    ]
    raise ValueError(
        f"Could not find label column. Tried {LABEL_CANDIDATES}. "
        f"Available label-like columns: {available_label_like_columns}"
    )

label_values = df[label_feature].astype("string").str.strip()
non_empty_label_mask = ~label_values.isna() & ~label_values.eq("")
missing_label_count = int((~non_empty_label_mask).sum())
unique_label_values = sorted(label_values[non_empty_label_mask].unique().tolist())
label_frequencies = label_values[non_empty_label_mask].value_counts().sort_values(ascending=False)

print()
print(f"Label feature analyzed: {label_feature}")
print(f"Unique label value count: {len(unique_label_values)}")
print("Label space (unique raw label values):")
for label_value in unique_label_values:
    print(f"- {label_value}")

print()
print("Label frequencies:")
for label_value, label_count in label_frequencies.items():
    print(f"- {label_value}: {int(label_count)}")

print(f"Missing or blank '{label_feature}' values: {missing_label_count}")

Feature/column count: 79
Feature space (all dataset columns):
-  Destination Port
-  Flow Duration
-  Total Fwd Packets
-  Total Backward Packets
- Total Length of Fwd Packets
-  Total Length of Bwd Packets
-  Fwd Packet Length Max
-  Fwd Packet Length Min
-  Fwd Packet Length Mean
-  Fwd Packet Length Std
- Bwd Packet Length Max
-  Bwd Packet Length Min
-  Bwd Packet Length Mean
-  Bwd Packet Length Std
- Flow Bytes/s
-  Flow Packets/s
-  Flow IAT Mean
-  Flow IAT Std
-  Flow IAT Max
-  Flow IAT Min
- Fwd IAT Total
-  Fwd IAT Mean
-  Fwd IAT Std
-  Fwd IAT Max
-  Fwd IAT Min
- Bwd IAT Total
-  Bwd IAT Mean
-  Bwd IAT Std
-  Bwd IAT Max
-  Bwd IAT Min
- Fwd PSH Flags
-  Bwd PSH Flags
-  Fwd URG Flags
-  Bwd URG Flags
-  Fwd Header Length
-  Bwd Header Length
- Fwd Packets/s
-  Bwd Packets/s
-  Min Packet Length
-  Max Packet Length
-  Packet Length Mean
-  Packet Length Std
-  Packet Length Variance
- FIN Flag Count
-  SYN Flag Count
-  RST Flag Count
-  PSH Flag Count
-  ACK Flag Coun

In [4]:
### Data sanitization ###

# Handle missing values by replacing all occurrences of infinity with NaN
# then removing rows containing NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# # remove duplicate flows and irrelevant columns
# df.drop_duplicates(inplace=True)
# df = df.drop(columns=["Flow ID", "Source IP", "Destination IP", "Timestamp"], errors="ignore")

# Remove leading/trailing spaces from all column names
df.rename(columns=lambda x: x.strip(), inplace=True)

In [5]:
### [Diagnostic] Extract features and labels ###

# Feature space: list every column present in the compiled source dataset.
feature_columns = list(df.columns)
print(f"Feature/column count: {len(feature_columns)}")
print("Feature space (all dataset columns):")
for column_name in feature_columns:
    print(f"- {column_name}")

# Label space: detect the raw label column robustly before sanitization trims names.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_feature = next((column_name for column_name in LABEL_CANDIDATES if column_name in df.columns), None)
if label_feature is None:
    available_label_like_columns = [
        column_name
        for column_name in df.columns
        if column_name.strip().lower() in {"label", "attack"}
    ]
    raise ValueError(
        f"Could not find label column. Tried {LABEL_CANDIDATES}. "
        f"Available label-like columns: {available_label_like_columns}"
    )

label_values = df[label_feature].astype("string").str.strip()
non_empty_label_mask = ~label_values.isna() & ~label_values.eq("")
missing_label_count = int((~non_empty_label_mask).sum())
unique_label_values = sorted(label_values[non_empty_label_mask].unique().tolist())
label_frequencies = label_values[non_empty_label_mask].value_counts().sort_values(ascending=False)

print()
print(f"Label feature analyzed: {label_feature}")
print(f"Unique label value count: {len(unique_label_values)}")
print("Label space (unique raw label values):")
for label_value in unique_label_values:
    print(f"- {label_value}")

print()
print("Label frequencies:")
for label_value, label_count in label_frequencies.items():
    print(f"- {label_value}: {int(label_count)}")

print(f"Missing or blank '{label_feature}' values: {missing_label_count}")

Feature/column count: 79
Feature space (all dataset columns):
- Destination Port
- Flow Duration
- Total Fwd Packets
- Total Backward Packets
- Total Length of Fwd Packets
- Total Length of Bwd Packets
- Fwd Packet Length Max
- Fwd Packet Length Min
- Fwd Packet Length Mean
- Fwd Packet Length Std
- Bwd Packet Length Max
- Bwd Packet Length Min
- Bwd Packet Length Mean
- Bwd Packet Length Std
- Flow Bytes/s
- Flow Packets/s
- Flow IAT Mean
- Flow IAT Std
- Flow IAT Max
- Flow IAT Min
- Fwd IAT Total
- Fwd IAT Mean
- Fwd IAT Std
- Fwd IAT Max
- Fwd IAT Min
- Bwd IAT Total
- Bwd IAT Mean
- Bwd IAT Std
- Bwd IAT Max
- Bwd IAT Min
- Fwd PSH Flags
- Bwd PSH Flags
- Fwd URG Flags
- Bwd URG Flags
- Fwd Header Length
- Bwd Header Length
- Fwd Packets/s
- Bwd Packets/s
- Min Packet Length
- Max Packet Length
- Packet Length Mean
- Packet Length Std
- Packet Length Variance
- FIN Flag Count
- SYN Flag Count
- RST Flag Count
- PSH Flag Count
- ACK Flag Count
- URG Flag Count
- CWE Flag Count
- EC

In [6]:
### Feature-space alignment ###
# (select features according to predetermined shared feature space)

# Canonical feature-space contract shared by source and target pipelines.
FEATURE_LIST_PATH = Path("data/processed/shared_feature_space.json")

# Handle missing file
if not FEATURE_LIST_PATH.exists():
    raise FileNotFoundError(
        f"Shared feature list not found at {FEATURE_LIST_PATH}. "
        "Create/populate this artifact before running preprocessing."
    )

# Load canonical ordered features. Order must be preserved.
shared_features = load_feature_order(FEATURE_LIST_PATH)

# Identify label column and handle leading or trailing whitespace
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# Aligns a DataFrame to the canonical shared feature contract by dropping extra
# columns, checking/optionally filling missing columns, and enforcing exact order.
# Used when source and target datasets must produce identical model input schema.
def align_feature_space(frame, feature_list, fill_missing=False, fill_value=0.0):
    
    # Compare incoming columns against the expected shared feature contract.
    feature_list = list(feature_list)
    incoming = set(frame.columns)
    expected = set(feature_list)

    extra = sorted(incoming - expected)
    missing = sorted(expected - incoming)

    # Strict mode (default): block pipeline if required features are absent.
    # This protects training/inference consistency across datasets.
    if missing and not fill_missing:
        preview = missing[:10]
        raise ValueError(
            f"Missing required features: {preview} (total={len(missing)})"
        )

    # Optional tolerant mode: create absent columns with a fixed value, then
    # continue with the canonical ordering.
    if missing and fill_missing:
        for col in missing:
            frame[col] = fill_value

    # Drop extras and enforce exact column order expected by downstream steps.
    aligned = frame[feature_list].copy()
    return aligned, extra, missing

# Ensure index is contiguous before splitting/rejoining features and labels.
df = df.reset_index(drop=True)

# Align only feature columns; label handling happens separately.
feature_df = df.drop(columns=[label_col]).copy()
aligned_X, dropped_extra, missing_cols = align_feature_space(
    feature_df,
    shared_features,
    fill_missing=False,
    fill_value=0.0,
 )

# Reattach labels by position (not index label) to avoid accidental NaNs.
labels_aligned = df[[label_col]].reset_index(drop=True)
aligned_X = aligned_X.reset_index(drop=True)
if len(aligned_X) != len(labels_aligned):
    raise ValueError(
        f"Feature/label row count mismatch after alignment: "
        f"X={len(aligned_X)}, y={len(labels_aligned)}"
    )
df = pd.concat([aligned_X, labels_aligned], axis=1)

print(f"Loaded shared feature list from {FEATURE_LIST_PATH}")
print(f"Aligned feature count: {len(shared_features)}")
print(f"Dropped extra columns: {len(dropped_extra)}")
print(f"Missing required columns: {len(missing_cols)}")
print(f"Missing labels after reattach: {int(df[label_col].isna().sum())}")

Loaded shared feature list from data/processed/shared_feature_space.json
Aligned feature count: 77
Dropped extra columns: 1
Missing required columns: 0
Missing labels after reattach: 0


In [7]:
### Label-space alignment ###
# (align labels according to predetermined shared label space)

SHARED_LABEL_SPACE_PATH = Path("data/processed/shared_label_space.json")
SOURCE_LABEL_MAP_PATH = Path("data/processed/source_label_map.json")

if not SHARED_LABEL_SPACE_PATH.exists():
    raise FileNotFoundError(
        f"Shared label space file not found at {SHARED_LABEL_SPACE_PATH}"
    )
if not SOURCE_LABEL_MAP_PATH.exists():
    raise FileNotFoundError(
        f"Source label map file not found at {SOURCE_LABEL_MAP_PATH}"
    )

# Load the canonical label contract in a validated format.
with open(SHARED_LABEL_SPACE_PATH, "r", encoding="utf-8") as f:
    shared_label_payload = json.load(f)

if isinstance(shared_label_payload, dict):
    if "labels" not in shared_label_payload:
        raise ValueError(
            f"Expected key 'labels' in {SHARED_LABEL_SPACE_PATH} when JSON object is provided."
        )
    shared_label_space = list(shared_label_payload["labels"])
elif isinstance(shared_label_payload, list):
    shared_label_space = list(shared_label_payload)
else:
    raise ValueError(
        f"Unsupported shared label space format in {SHARED_LABEL_SPACE_PATH}: "
        f"{type(shared_label_payload).__name__}"
    )

if not shared_label_space:
    raise ValueError(f"Shared label space in {SHARED_LABEL_SPACE_PATH} is empty")
if len(shared_label_space) != len(set(shared_label_space)):
    duplicate_labels = sorted(
        label for label in set(shared_label_space) if shared_label_space.count(label) > 1
    )
    raise ValueError(
        f"Duplicate canonical labels found in {SHARED_LABEL_SPACE_PATH}: {duplicate_labels}"
    )

with open(SOURCE_LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    source_label_map = json.load(f)
if not isinstance(source_label_map, dict):
    raise ValueError(
        f"Expected JSON object at {SOURCE_LABEL_MAP_PATH}, got "
        f"{type(source_label_map).__name__}"
    )

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# Normalize raw labels for stable matching (trim spaces, keep missing as <NA>).
raw_labels = df[label_col].astype("string").str.strip()

# Detect genuinely missing labels (null/blank) separately from unmapped labels.
missing_label_mask = raw_labels.isna() | raw_labels.eq("")
missing_label_count = int(missing_label_mask.sum())

if missing_label_count > 0:
    missing_indices = df.index[missing_label_mask].tolist()
    preview_rows = missing_indices[:10]
    preview_values = [repr(v) for v in raw_labels.loc[preview_rows].tolist()]
    raise ValueError(
        f"Missing label values found at row indices {preview_rows} "
        f"(total={missing_label_count}). "
        f"Sample raw values at those rows: {preview_values}. "
        "Clean/drop these rows before label alignment."
    )

# Guardrail: mapping file must only map into allowed shared classes.
invalid_canonical_labels = sorted(
    set(source_label_map.values()) - set(shared_label_space)
)
if invalid_canonical_labels:
    raise ValueError(
        "source_label_map.json contains classes not present in shared_label_space.json: "
        f"{invalid_canonical_labels}"
    )

# Apply raw->shared mapping.
mapped_labels = raw_labels.map(source_label_map)

# Drop unmapped non-missing raw labels by design instead of crashing the pipeline.
unmapped_mask = (~missing_label_mask) & mapped_labels.isna()
dropped_unmapped_count = int(unmapped_mask.sum())
if dropped_unmapped_count > 0:
    unmapped_raw = raw_labels[unmapped_mask]
    unmapped_counts = unmapped_raw.value_counts().sort_values(ascending=False)
    keep_mask = ~unmapped_mask

    print("Dropping unmapped raw labels from source dataset:")
    for raw_label, raw_count in unmapped_counts.items():
        print(f"- {raw_label}: {int(raw_count)}")

    df = df.loc[keep_mask].reset_index(drop=True)
    mapped_labels = mapped_labels.loc[keep_mask].reset_index(drop=True)
else:
    df = df.reset_index(drop=True)
    mapped_labels = mapped_labels.reset_index(drop=True)

if df.empty:
    raise ValueError(
        "All rows were removed during source label alignment. "
        f"Check {SOURCE_LABEL_MAP_PATH} and {SHARED_LABEL_SPACE_PATH}."
    )

# Replace dataset labels with aligned shared classes.
df[label_col] = mapped_labels

# Sanity check: print class-frequency table after label alignment.
label_counts = df[label_col].value_counts(dropna=False).sort_values(ascending=False)
label_freq = (label_counts / len(df) * 100).round(2)

print(f"Loaded shared label space from {SHARED_LABEL_SPACE_PATH}")
print(f"Loaded source label map from {SOURCE_LABEL_MAP_PATH}")
print(f"Canonical label count: {len(shared_label_space)}")
print(f"Dropped unmapped rows: {dropped_unmapped_count}")
print(f"Rows retained after label alignment: {len(df)}")
print("Label category frequencies after alignment:")
for cls in label_counts.index:
    print(f"- {cls}: {int(label_counts[cls])} ({label_freq[cls]:.2f}%)")

Dropping unmapped raw labels from source dataset:
- PortScan: 158804
- Bot: 1956
- Web Attack � Brute Force: 1507
- Web Attack � XSS: 652
- Infiltration: 36
- Web Attack � Sql Injection: 21
- Heartbleed: 11
Loaded shared label space from data/processed/shared_label_space.json
Loaded source label map from data/processed/source_label_map.json
Canonical label count: 8
Dropped unmapped rows: 162987
Rows retained after label alignment: 2664889
Label category frequencies after alignment:
- Benign: 2271320 (85.23%)
- DoS Hulk: 230124 (8.64%)
- DDoS: 128025 (4.80%)
- DoS GoldenEye: 10293 (0.39%)
- FTP-Patator: 7935 (0.30%)
- SSH-Patator: 5897 (0.22%)
- DoS slowloris: 5796 (0.22%)
- DoS Slowhttptest: 5499 (0.21%)


In [8]:
### Standardize benign/attack proportion ###
# Target: 75% benign / 25% attack.
# All attack samples are preserved; benign samples are randomly downsampled.

TARGET_BENIGN_RATIO = 0.75  # fraction of total that should be benign

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

BENIGN_LABEL = "Benign"

benign_mask = df[label_col].astype("string").str.strip().str.upper() == BENIGN_LABEL.upper()
attack_mask = ~benign_mask

n_benign = int(benign_mask.sum())
n_attack = int(attack_mask.sum())

print(f"Before resampling: benign={n_benign}, attack={n_attack}, total={len(df)}")

# With all attacks preserved, required benign count for target ratio:
# n_benign_target / (n_benign_target + n_attack) = TARGET_BENIGN_RATIO
# => n_benign_target = n_attack * TARGET_BENIGN_RATIO / (1 - TARGET_BENIGN_RATIO)
n_benign_target = int(round(n_attack * TARGET_BENIGN_RATIO / (1.0 - TARGET_BENIGN_RATIO)))

if n_benign_target >= n_benign:
    print(
        f"Current benign count ({n_benign}) is already at or below the target ({n_benign_target}). "
        "No downsampling needed."
    )
else:
    benign_idx = df.index[benign_mask]
    kept_benign_idx = benign_idx.to_series().sample(n=n_benign_target, random_state=42).index
    attack_idx = df.index[attack_mask]
    df = df.loc[kept_benign_idx.union(attack_idx)].sample(frac=1, random_state=42).reset_index(drop=True)

    n_benign_after = int((df[label_col].astype("string").str.strip().str.upper() == BENIGN_LABEL.upper()).sum())
    n_attack_after = len(df) - n_benign_after
    actual_ratio = n_benign_after / len(df) * 100

    print(f"After resampling:  benign={n_benign_after}, attack={n_attack_after}, total={len(df)}")
    print(f"Benign proportion: {actual_ratio:.2f}% (target: {TARGET_BENIGN_RATIO * 100:.0f}%)")



Before resampling: benign=2271320, attack=393569, total=2664889
After resampling:  benign=1180707, attack=393569, total=1574276
Benign proportion: 75.00% (target: 75%)


In [9]:
### Train/Test Split ###

from sklearn.model_selection import train_test_split

# Re-detect label column to keep this cell independently runnable.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_col = next((c for c in LABEL_CANDIDATES if c in df.columns), None)
if label_col is None:
    raise ValueError(f"Could not find label column. Tried {LABEL_CANDIDATES}")

# Create feature matrix X and target vector y.
X = df.drop(columns=[label_col])
y = df[label_col]

# Single split: 80% train, 20% test.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
    shuffle=True,
 )

print(f"Train/Test sizes: {len(y_train)}/{len(y_test)}")

Train/Test sizes: 1259420/314856


In [10]:
### Scaling (save scaler for target data processing) ###
# Note: applies scaler to data; exported data is already scaled

# Fit (and apply) scaler on training split only to avoid data leakage, then
# apply to test split.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames to preserve feature names/indexing.
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

# Save fitted scaler for reuse in target-domain preprocessing/inference.
scaler_path = Path("models/source_scaler.joblib")
scaler_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, scaler_path)

print(f"Saved scaler to {scaler_path}")
print(f"Scaled splits shapes: train={X_train.shape}, test={X_test.shape}")

Saved scaler to models/source_scaler.joblib
Scaled splits shapes: train=(1259420, 77), test=(314856, 77)


In [11]:
### Label encoding (save encoder for target data processing) ###

# Fit (and apply) label encoder on training labels and apply to test labels.
le = LabelEncoder()
y_train = pd.Series(le.fit_transform(y_train), index=y_train.index, name=label_col)
y_test = pd.Series(le.transform(y_test), index=y_test.index, name=label_col)

# Save fitted label encoder for reuse in target-domain preprocessing/inference.
encoder_path = Path("models/label_encoder.joblib")
encoder_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(le, encoder_path)

print(f"Saved label encoder to {encoder_path}")
print(f"Encoded classes ({len(le.classes_)}): {list(le.classes_)}")

Saved label encoder to models/label_encoder.joblib
Encoded classes (8): ['Benign', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'SSH-Patator']


In [12]:
# ### [DEPRECATED; moved] Extract global CORAL statistics ###
# # Note: calculated from training split only.

# # Load shared feature space contract.
# shared_feature_space_path = Path("data/processed/shared_feature_space.json")
# if not shared_feature_space_path.exists():
#     raise FileNotFoundError(f"Shared feature space file not found at {shared_feature_space_path}")

# # Reuse unified parser so feature-space JSON is handled consistently across cells.
# feature_order = load_feature_order(shared_feature_space_path)

# # Improvement support: persist canonical feature ordering inside CORAL stats so
# # evaluation can verify schema compatibility before adaptation.
# X_train_aligned = X_train[feature_order]

# # Convert to numpy for CORAL math using float64 for better numerical stability.
# X_src = X_train_aligned.to_numpy(dtype=np.float64)

# # 1) Feature-wise mean vector.
# source_feature_mean = np.mean(X_src, axis=0)

# # 2) Centered source data.
# X_src_centered = X_src - source_feature_mean

# # Improvement #1: use Ledoit-Wolf shrinkage covariance for a better-conditioned
# # covariance estimate than plain sample covariance.
# source_cov_estimator = LedoitWolf()
# source_cov_estimator.fit(X_src_centered)
# source_covariance = np.asarray(source_cov_estimator.covariance_, dtype=np.float64)
# source_covariance = (source_covariance + source_covariance.T) / 2.0

# # Save diagnostics consumed by training/eval for spectral-floor and stability analysis.
# source_eigenvalues = np.linalg.eigvalsh(source_covariance)
# source_min_eig = float(source_eigenvalues.min())
# source_max_eig = float(source_eigenvalues.max())
# source_cov_condition_number = float(np.linalg.cond(source_covariance))
# source_covariance_ridge = 0.0

# # Sanity checks.
# assert source_covariance.shape[0] == source_covariance.shape[1], "Covariance matrix must be square"
# assert source_covariance.shape[0] == len(feature_order), "Covariance dimension mismatch with feature space"

# # Package CORAL statistics.
# coral_source_stats = {
#     "feature_order": feature_order,
#     "mean": source_feature_mean,
#     "covariance": source_covariance,
#     "covariance_estimator": "LedoitWolf",
#     "covariance_shrinkage": float(source_cov_estimator.shrinkage_),
#     "min_eigenvalue_before_regularization": source_min_eig,
#     "max_eigenvalue": source_max_eig,
#     "covariance_condition_number": source_cov_condition_number,
#     "covariance_ridge": source_covariance_ridge,
# }

# # Persist for downstream domain adaptation pipeline.
# coral_stats_path = Path("models/coral_source_stats.joblib")
# coral_stats_path.parent.mkdir(parents=True, exist_ok=True)
# joblib.dump(coral_source_stats, coral_stats_path)

# print("CORAL source statistics extracted and saved successfully.")
# print(f"Saved to: {coral_stats_path}")
# print(f"Features: {len(feature_order)}")
# print(f"Covariance shape: {source_covariance.shape}")
# print(f"Covariance estimator: LedoitWolf (shrinkage={source_cov_estimator.shrinkage_:.6f})")
# print(f"Covariance eigenvalues: min={source_min_eig:.6e}, max={source_max_eig:.6e}")
# print(f"Covariance condition number: {source_cov_condition_number:.6e}")

In [13]:
# ### [DEPRECATED; moved] Extract per-class CORAL statistics ###
# # Note: calculated from training split only, separately for each class.
# # Exports to "models/perclass_coral_source_stats.joblib"

# perclass_stats = {}
# classes_in_train = sorted(pd.Series(y_train).unique().tolist())

# if len(classes_in_train) == 0:
#     raise ValueError("No classes found in y_train; cannot compute per-class CORAL statistics")

# for class_id in classes_in_train:
#     class_mask = (y_train == class_id)
#     class_count = int(class_mask.sum())
#     if class_count < 2:
#         raise ValueError(
#             f"Class {class_id} has only {class_count} sample(s) in training split. "
#             "At least 2 samples are required to estimate covariance."
#         )

#     # Keep exact same feature ordering contract as global CORAL statistics.
#     X_class = X_train.loc[class_mask, feature_order]
#     X_class_np = X_class.to_numpy(dtype=np.float64)

#     # 1) Feature-wise mean vector for this class.
#     class_feature_mean = np.mean(X_class_np, axis=0)

#     # 2) Center class data.
#     X_class_centered = X_class_np - class_feature_mean

#     # 3) Ledoit-Wolf covariance estimate (same estimator as global cell).
#     class_cov_estimator = LedoitWolf()
#     class_cov_estimator.fit(X_class_centered)
#     class_covariance = np.asarray(class_cov_estimator.covariance_, dtype=np.float64)
#     class_covariance = (class_covariance + class_covariance.T) / 2.0

#     # Diagnostics mirroring global-stat extraction.
#     class_eigenvalues = np.linalg.eigvalsh(class_covariance)
#     class_min_eig = float(class_eigenvalues.min())
#     class_max_eig = float(class_eigenvalues.max())
#     class_cov_condition_number = float(np.linalg.cond(class_covariance))
#     class_covariance_ridge = 0.0

#     assert class_covariance.shape[0] == class_covariance.shape[1], "Covariance matrix must be square"
#     assert class_covariance.shape[0] == len(feature_order), "Covariance dimension mismatch with feature space"

#     class_name = str(class_id)
#     if "le" in globals() and hasattr(le, "classes_"):
#         # Prefer original class label when encoder is available.
#         class_name = str(le.inverse_transform([int(class_id)])[0])

#     perclass_stats[class_name] = {
#         "class_id": int(class_id),
#         "class_name": class_name,
#         "sample_count": class_count,
#         "feature_order": feature_order,
#         "mean": class_feature_mean,
#         "covariance": class_covariance,
#         "covariance_estimator": "LedoitWolf",
#         "covariance_shrinkage": float(class_cov_estimator.shrinkage_),
#         "min_eigenvalue_before_regularization": class_min_eig,
#         "max_eigenvalue": class_max_eig,
#         "covariance_condition_number": class_cov_condition_number,
#         "covariance_ridge": class_covariance_ridge,
#     }

# # Persist per-class stats for downstream class-conditional CORAL variants.
# perclass_stats_path = Path("models/perclass_coral_source_stats.joblib")
# perclass_stats_path.parent.mkdir(parents=True, exist_ok=True)
# joblib.dump(perclass_stats, perclass_stats_path)

# print("Per-class CORAL source statistics extracted and saved successfully.")
# print(f"Saved to: {perclass_stats_path}")
# print(f"Classes processed: {len(perclass_stats)}")
# for class_name, stats in perclass_stats.items():
#     print(
#         f"- {class_name}: n={stats['sample_count']}, "
#         f"cov_shape={stats['covariance'].shape}, "
#         f"shrinkage={stats['covariance_shrinkage']:.6f}, "
#         f"min_eig={stats['min_eigenvalue_before_regularization']:.6e}, "
#         f"cond={stats['covariance_condition_number']:.6e}"
#     )

In [14]:
### [Diagnostic] Extract features and labels ###
# Note: final verification of alignment with shared feature and label spaces

# Feature space: list every column present in the compiled source dataset.
feature_columns = list(df.columns)
print(f"Feature/column count: {len(feature_columns)}")
print("Feature space (all dataset columns):")
for column_name in feature_columns:
    print(f"- {column_name}")

# Label space: detect the raw label column robustly before sanitization trims names.
LABEL_CANDIDATES = ["Label", " Label", "label"]
label_feature = next((column_name for column_name in LABEL_CANDIDATES if column_name in df.columns), None)
if label_feature is None:
    available_label_like_columns = [
        column_name
        for column_name in df.columns
        if column_name.strip().lower() in {"label", "attack"}
    ]
    raise ValueError(
        f"Could not find label column. Tried {LABEL_CANDIDATES}. "
        f"Available label-like columns: {available_label_like_columns}"
    )

label_values = df[label_feature].astype("string").str.strip()
non_empty_label_mask = ~label_values.isna() & ~label_values.eq("")
missing_label_count = int((~non_empty_label_mask).sum())
unique_label_values = sorted(label_values[non_empty_label_mask].unique().tolist())
label_frequencies = label_values[non_empty_label_mask].value_counts().sort_values(ascending=False)

print()
print(f"Label feature analyzed: {label_feature}")
print(f"Unique label value count: {len(unique_label_values)}")
print("Label space (unique raw label values):")
for label_value in unique_label_values:
    print(f"- {label_value}")

print()
print("Label frequencies:")
for label_value, label_count in label_frequencies.items():
    print(f"- {label_value}: {int(label_count)}")

print(f"Missing or blank '{label_feature}' values: {missing_label_count}")

Feature/column count: 78
Feature space (all dataset columns):
- Destination Port
- Flow Duration
- Total Fwd Packets
- Total Backward Packets
- Total Length of Fwd Packets
- Total Length of Bwd Packets
- Fwd Packet Length Max
- Fwd Packet Length Min
- Fwd Packet Length Mean
- Fwd Packet Length Std
- Bwd Packet Length Max
- Bwd Packet Length Min
- Bwd Packet Length Mean
- Bwd Packet Length Std
- Flow Bytes/s
- Flow Packets/s
- Flow IAT Mean
- Flow IAT Std
- Flow IAT Max
- Flow IAT Min
- Fwd IAT Total
- Fwd IAT Mean
- Fwd IAT Std
- Fwd IAT Max
- Fwd IAT Min
- Bwd IAT Total
- Bwd IAT Mean
- Bwd IAT Std
- Bwd IAT Max
- Bwd IAT Min
- Fwd PSH Flags
- Bwd PSH Flags
- Fwd URG Flags
- Bwd URG Flags
- Fwd Header Length
- Bwd Header Length
- Fwd Packets/s
- Bwd Packets/s
- Min Packet Length
- Max Packet Length
- Packet Length Mean
- Packet Length Std
- Packet Length Variance
- FIN Flag Count
- SYN Flag Count
- RST Flag Count
- PSH Flag Count
- ACK Flag Count
- URG Flag Count
- CWE Flag Count
- EC


Label feature analyzed: Label
Unique label value count: 8
Label space (unique raw label values):
- Benign
- DDoS
- DoS GoldenEye
- DoS Hulk
- DoS Slowhttptest
- DoS slowloris
- FTP-Patator
- SSH-Patator

Label frequencies:
- Benign: 1180707
- DoS Hulk: 230124
- DDoS: 128025
- DoS GoldenEye: 10293
- FTP-Patator: 7935
- SSH-Patator: 5897
- DoS slowloris: 5796
- DoS Slowhttptest: 5499
Missing or blank 'Label' values: 0


In [15]:
### Export processed data ###

# Create output directory for processed source splits.
output_dir = Path("data/processed/source")
output_dir.mkdir(parents=True, exist_ok=True)

# Save training data.
train_df = pd.DataFrame(X_train, columns=X_train.columns)
train_df["Label"] = y_train.values
train_df.to_csv(output_dir / "source_train.csv", index=False)

# Save test data.
test_df = pd.DataFrame(X_test, columns=X_test.columns)
test_df["Label"] = y_test.values
test_df.to_csv(output_dir / "source_test.csv", index=False)

print("Saved datasets:")
print(f"  Train: {len(y_train)} samples")
print(f"  Test: {len(y_test)} samples")
print(f"  Output directory: {output_dir}")

Saved datasets:
  Train: 1259420 samples
  Test: 314856 samples
  Output directory: data/processed/source
